# Notebook 03 — Training the GNN

Notebook 02 turned every C/C++ function into a PyTorch Geometric `Data` object and saved the three splits to `data/processed/{train,valid,test}.pt`. This notebook **trains** the `Vuln47GNN` on those tensors.

As with the earlier notebooks, no logic is duplicated here: the training loop lives entirely in the `source/training/` package, and this notebook simply wires it together and explains each decision. The orchestration is done by `ModelTrainingPipeline`, which for every epoch:

1. trains one pass over the `train` split (`ModelTrainer`, class-weighted loss),
2. evaluates on the `valid` split (`MetricsEvaluator`),
3. keeps the checkpoint with the best validation **PR-AUC**,

then restores that best checkpoint and reports its `test` metrics.

**What this notebook covers:**
1. Setup and the training configuration
2. The batched data loaders
3. Handling class imbalance — the weighted loss
4. The training loop (running the pipeline)
5. Reading the training result and the saved checkpoint

> **Note on runtime.** The `train` split is ~175k graphs; a full 30-epoch run on the M4 Pro's MPS backend takes a while. The heavy cell is clearly marked so it can be run deliberately.

## 1. Setup & Training Configuration

As in the other notebooks, the project root is put on `sys.path` so the `source.` package resolves from `notebooks/`. Then the whole training configuration is imported from `training_config.py` — a single place that fixes the device, the split paths, the optimizer hyperparameters, and where the best checkpoint is written. Printing it here makes the run reproducible and self-documenting.

In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# training must run from the project root so the data/ paths in training_config resolve
os.chdir(PROJECT_ROOT)

import torch

from source.training.model_training import training_config as cfg
from source.preprocessing.data_preprocessing.data_representation.data_node_representation.code_node_representator import (
    CodeNodeRepresentator,
)

print("Training configuration")
print(f"  device          : {cfg.DEVICE}")
print(f"  train / valid / test:")
print(f"      {cfg.TRAIN_PATH}")
print(f"      {cfg.VALID_PATH}")
print(f"      {cfg.TEST_PATH}")
print(f"  checkpoint      : {cfg.CHECKPOINT_PATH}")
print(f"  epochs          : {cfg.EPOCHS}")
print(f"  batch size      : {cfg.BATCH_SIZE}")
print(f"  learning rate   : {cfg.LR}")
print(f"  weight decay    : {cfg.WEIGHT_DECAY}")
print(f"  best metric     : {cfg.BEST_METRIC}")

vocab = CodeNodeRepresentator.load_or_build_vocab()
print(f"\nVocabulary: {len(vocab)} AST node types (sizes the type-embedding table)")

Training configuration
  device          : mps
  train / valid / test:
      data/processed/train.pt
      data/processed/valid.pt
      data/processed/test.pt
  checkpoint      : data/model/vuln_gnn.pt
  epochs          : 30
  batch size      : 64
  learning rate   : 0.001
  weight decay    : 1e-05
  best metric     : pr_auc

Vocabulary: 112 AST node types (sizes the type-embedding table)


## 2. Batched Data Loaders

A GNN cannot consume 175k graphs at once; they are grouped into **mini-batches**. `GraphBatchLoader` wraps PyTorch Geometric's `DataLoader`, which does something clever for graphs: it merges the graphs in a batch into **one big disconnected graph** and adds a `batch` vector recording which node belongs to which original graph. Pooling later uses that vector to collapse each graph back to a single vector.

The `train` loader shuffles each epoch (so batch composition varies); the `valid`/`test` loaders do not (evaluation must be deterministic). The cell below builds the loaders and peeks at one batch to make the batching concrete.

In [2]:
from source.training.model_training.batch_loading.graph_batch_loader import GraphBatchLoader

train_loader = GraphBatchLoader(shuffle=True).load(cfg.TRAIN_PATH)
valid_loader = GraphBatchLoader(shuffle=False).load(cfg.VALID_PATH)

print(f"train batches : {len(train_loader)}  (batch_size={cfg.BATCH_SIZE})")
print(f"valid batches : {len(valid_loader)}")

# inspect a single batch to see how PyG merges graphs
sample_batch = next(iter(valid_loader))
print(f"\nOne batch:")
print(f"  {sample_batch}")
print(f"  graphs in batch : {sample_batch.num_graphs}")
print(f"  total nodes     : {sample_batch.x.shape[0]}")
print(f"  x               : {tuple(sample_batch.x.shape)}  (6 features per node)")
print(f"  edge_index      : {tuple(sample_batch.edge_index.shape)}")
print(f"  batch vector    : {tuple(sample_batch.batch.shape)}  "
      f"(maps each node -> its graph id, values 0..{sample_batch.num_graphs - 1})")

train batches : 2874  (batch_size=64)
valid batches : 397

One batch:
  DataBatch(x=[4835, 7], edge_index=[2, 15224], edge_attr=[15224], y=[64], batch=[4835], ptr=[65])
  graphs in batch : 64
  total nodes     : 4835
  x               : (4835, 7)  (6 features per node)
  edge_index      : (2, 15224)
  batch vector    : (4835,)  (maps each node -> its graph id, values 0..63)


## 3. Handling Class Imbalance — The Weighted Loss

Notebook 01 established the defining property of the data: only **~2.7%** of `train` functions are vulnerable. Left unaddressed, the optimizer would minimise loss simply by predicting "safe" for everything.

The counter-measure is a **class-weighted cross-entropy loss**. The pipeline computes a weight vector `[1.0, n_safe / n_vuln]` — so an error on the rare *vulnerable* class costs roughly `n_safe / n_vuln ≈ 35×` more than an error on the *safe* class. This pushes the model to actually attend to vulnerabilities. The exact computation done inside `ModelTrainingPipeline._compute_class_weight` is reproduced below so the number is visible.

In [3]:
n_vuln = n_safe = 0
for batch in train_loader:
    y = batch.y.view(-1)
    n_vuln += int((y == 1).sum())
    n_safe += int((y == 0).sum())

ratio = n_safe / max(n_vuln, 1)
class_weight = torch.tensor([1.0, ratio], dtype=torch.float)

print(f"train: {n_safe:,} safe  vs  {n_vuln:,} vulnerable")
print(f"class weight [safe, vuln] = [1.0, {ratio:.1f}]")
print(f"\n-> an error on a vulnerable function is penalised {ratio:.0f}x more heavily,")
print(f"   which is how the model is stopped from collapsing to 'always safe'.")

train: 178,481 safe  vs  5,417 vulnerable
class weight [safe, vuln] = [1.0, 32.9]

-> an error on a vulnerable function is penalised 33x more heavily,
   which is how the model is stopped from collapsing to 'always safe'.


## 4. The Training Loop

Everything now comes together in `ModelTrainingPipeline.run()`. Given only the vocabulary, it internally:

- builds the loaders (§2) and the `Vuln47GNN` model,
- computes the class weight (§3) and an Adam optimizer,
- for each of `EPOCHS` epochs: trains one pass, evaluates on `valid`, and **remembers the state dict with the best validation PR-AUC**,
- writes that best checkpoint to `CHECKPOINT_PATH`,
- reloads it and evaluates once on the held-out `test` split,

returning a dictionary with the best epoch, the best validation metrics, the final test metrics, and the checkpoint path.

> ⚠️ **This is the heavy, long-running cell.** It trains on ~175k graphs for `EPOCHS` epochs on MPS. Run it deliberately; progress bars report per-epoch train/eval progress, and each epoch prints its loss and validation metrics.

**Resume-from-interrupt.** Because a full run takes hours, the loop is now crash-safe: after **every** epoch the pipeline writes the complete training state (model, optimizer, LR scheduler, early-stopper counter, best-so-far, and RNG state) to a rolling checkpoint at `RESUME_PATH` (`data/model/last.pt`). If the run is interrupted — kernel crash, Colab timeout, manual stop — just **re-run this cell**: it auto-detects `last.pt`, prints `resuming from epoch N`, and continues exactly where it left off. On a clean finish the pipeline deletes `last.pt`, so the next run starts fresh. To force a from-scratch run, delete `data/model/last.pt` manually before running.

In [4]:
from source.training.model_training.model_training_pipeline import ModelTrainingPipeline

pipeline = ModelTrainingPipeline(vocab)
result = pipeline.run()

train: 100%|██████████| 2874/2874 [09:15<00:00,  5.17it/s]


epoch 0: loss=0.7998 f1=0.0970 precision=0.0514 recall=0.8536 pr_auc=0.1030 lr=1.00e-03


train: 100%|██████████| 2874/2874 [14:39<00:00,  3.27it/s]


epoch 1: loss=0.7010 f1=0.1386 precision=0.0782 recall=0.6076 pr_auc=0.1094 lr=1.00e-03


train: 100%|██████████| 2874/2874 [17:10<00:00,  2.79it/s]


...

epoch 9: loss=0.5653 f1=0.1160 precision=0.0627 recall=0.7731 pr_auc=0.1416 lr=1.00e-03


train: 100%|██████████| 2874/2874 [17:39<00:00,  2.71it/s]


epoch 10: loss=0.5491 f1=0.1089 precision=0.0585 recall=0.7862 pr_auc=0.1490 lr=1.00e-03


train:  98%|█████████▊| 2808/2874 [17:28<01:18,  1.18s/it]

: 

## 5. Training Result & Saved Checkpoint

The dictionary returned by the pipeline summarises the run: which epoch was best on `valid`, that epoch's validation metrics, and the corresponding `test` metrics. The best checkpoint has been written to disk — its existence and size are confirmed below. **Notebook 04** loads exactly this file to perform a deeper evaluation (confusion matrix, PR curve, threshold analysis).

In [5]:
def _fmt(metrics):
    return "  ".join(f"{k}={v:.4f}" for k, v in metrics.items())

print(f"best epoch (by valid {cfg.BEST_METRIC}): {result['best_epoch']}\n")
print(f"valid @ best : {_fmt(result['best_valid'])}")
print(f"test  @ best : {_fmt(result['test'])}")

ckpt = result["checkpoint"]
if os.path.exists(ckpt):
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f"\ncheckpoint written: {ckpt}  ({size_mb:.2f} MB)")
else:
    print(f"\nWARNING: expected checkpoint at {ckpt} was not found.")

best epoch (by valid pr_auc): 21

valid @ best : f1=0.1668  precision=0.0967  recall=0.6076  pr_auc=0.1374
test  @ best : f1=0.1617  precision=0.0926  recall=0.6379  pr_auc=0.1260

checkpoint written: data/model/vuln_gnn.pt  (0.75 MB)
